# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hardik144/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule: Low CTR despite good search visibility

I will prioritize content items that receive a reasonable number of impressions and appear in good Google search positions but have a comparatively low click-through rate (CTR).

The rule will use historical search-performance signals available within the development window. Each qualifying content item will receive a score, one reason code, and an action label.

**Reason code:** `LOW_CTR_GOOD_POSITION`

**Action label:** `REVIEW_TITLE_META`

The thresholds will be selected after examining the actual data distributions. This is a prioritization rule for human review, not a guarantee that changing a title or meta description will improve performance.


In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [18]:

# SECTION 2: BUILD THE RANKED QUEUE

import pandas as pd
import numpy as np
from pathlib import Path

# Work with a copy of the existing dataset
ranked_df = df.copy()

# Create one simple baseline rule:
# Prioritize pages ranking in positions 4-20
# that have below-median CTR and at least some impressions.

median_ctr = ranked_df["ctr"].median()

eligible = (
    ranked_df["avg_position"].between(4, 20, inclusive="both")
    & (ranked_df["ctr"] < median_ctr)
    & (ranked_df["impressions_90d"] > 0)
)

ranked_df["score"] = np.where(
    eligible,
    (21 - ranked_df["avg_position"]) * 10
    + (ranked_df["impressions_90d"] / 1000),
    0
)

ranked_df["reason_code"] = np.where(
    eligible,
    "CTR_LOW_POS_4_20",
    "NOT_ELIGIBLE"
)

ranked_df["action"] = np.where(
    eligible,
    "REVIEW_TITLE_SNIPPET",
    "NO_ACTION"
)

# Rank the recommendations
ranked_df = ranked_df.sort_values(
    "score", ascending=False
).reset_index(drop=True)

ranked_df["rank"] = ranked_df.index + 1

# Keep the recommended actions in the queue
ranked_df = ranked_df[ranked_df["score"] > 0].copy()

# Write the CSV output
output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

ranked_df.to_csv(output_path, index=False)

print("Ranked queue created successfully!")
print("Number of recommendations:", len(ranked_df))
print("CSV saved to:", output_path)

display(ranked_df.head(20))

Ranked queue created successfully!
Number of recommendations: 7395
CSV saved to: work/outputs/baseline_action_score.csv


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,score,reason_code,action,rank
0,content_36ff89c8214e,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,4.48,0.00,excellent,page_1,stable,0.5,432.097,CTR_LOW_POS_4_20,REVIEW_TITLE_SNIPPET,1
1,content_c84a0ab98e90,client_f369cb89fc,0.0,0.00,LOW,0.00,keyword article,informational,2871.0,19536.0,...,25.00,0.00,excellent,page_1,stable,17.2,355.271,CTR_LOW_POS_4_20,REVIEW_TITLE_SNIPPET,2
2,content_c8e9d6ab9013,client_19581e27de,20.0,0.03,LOW,0.00,keyword article,informational,NaN,NaN,...,0.00,0.00,excellent,page_1,down,-43.4,321.678,CTR_LOW_POS_4_20,REVIEW_TITLE_SNIPPET,3
3,content_91652435f57a,client_19581e27de,10.0,0.29,LOW,0.09,keyword article,commercial,NaN,NaN,...,2.56,0.00,excellent,page_1,stable,-1.4,291.590,CTR_LOW_POS_4_20,REVIEW_TITLE_SNIPPET,4
4,content_453722754fea,client_f369cb89fc,10.0,0.00,LOW,0.00,keyword article,informational,2700.0,18723.0,...,11.11,0.00,excellent,page_1,down,-52.9,274.079,CTR_LOW_POS_4_20,REVIEW_TITLE_SNIPPET,5
5,content_c1fe78bc4e37,client_19581e27de,70.0,0.04,LOW,0.00,keyword article,commercial,NaN,NaN,...,4.64,0.00,excellent,page_1,down,-41.4,269.055,CTR_LOW_POS_4_20,REVIEW_TITLE_SNIPPET,6
6,content_0919dd345d80,client_4e07408562,40.0,0.00,LOW,0.00,keyword article,informational,2841.0,18175.0,...,8.70,0.00,excellent,page_1,down,-76.1,259.217,CTR_LOW_POS_4_20,REVIEW_TITLE_SNIPPET,7
7,content_b115f7c74779,client_19581e27de,30.0,0.02,LOW,0.00,keyword article,transactional,NaN,NaN,...,4.08,0.00,excellent,page_1,up,35.9,253.469,CTR_LOW_POS_4_20,REVIEW_TITLE_SNIPPET,8
8,content_39881853ef0c,client_f369cb89fc,170.0,0.02,LOW,0.00,keyword article,informational,2810.0,19393.0,...,15.38,0.00,excellent,page_1,down,-42.0,250.434,CTR_LOW_POS_4_20,REVIEW_TITLE_SNIPPET,9
9,content_63f88d16fdb8,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,9.42,0.00,excellent,page_1,down,-37.8,245.013,CTR_LOW_POS_4_20,REVIEW_TITLE_SNIPPET,10


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 Review

I reviewed the 20 highest-ranked items produced by my baseline rule. For each item, I recorded the recommended action, reason code, confidence note, and a condition that could make the recommendation incorrect.

The ranking is intended for decision support, not as proof that an action will improve performance. The recommendations are based only on the observed dataset signals and do not use future-window or label-derived information.


In [19]:
# SECTION 3: TOP-20 REVIEW

# Use the ranked queue created in Section 2.
# Change ranked_df if your queue has a different variable name.

top20 = ranked_df.head(20).copy()

# Add review notes for each recommendation.
top20["confidence_note"] = (
    "Baseline heuristic; requires manual validation."
)

top20["what_would_make_it_wrong"] = (
    "The observed signal may be noisy, incomplete, "
    "or explained by factors not included in the rule."
)

# Display the review.
review_columns = [
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

print("TOP-20 REVIEW")
display(top20[review_columns])

TOP-20 REVIEW


,action,reason_code,confidence_note,what_would_make_it_wrong
0,REVIEW_TITLE_SNIPPET,CTR_LOW_POS_4_20,Baseline heuristic; requires manual validation.,"The observed signal may be noisy, incomplete, ..."
1,REVIEW_TITLE_SNIPPET,CTR_LOW_POS_4_20,Baseline heuristic; requires manual validation.,"The observed signal may be noisy, incomplete, ..."
2,REVIEW_TITLE_SNIPPET,CTR_LOW_POS_4_20,Baseline heuristic; requires manual validation.,"The observed signal may be noisy, incomplete, ..."
3,REVIEW_TITLE_SNIPPET,CTR_LOW_POS_4_20,Baseline heuristic; requires manual validation.,"The observed signal may be noisy, incomplete, ..."
4,REVIEW_TITLE_SNIPPET,CTR_LOW_POS_4_20,Baseline heuristic; requires manual validation.,"The observed signal may be noisy, incomplete, ..."
5,REVIEW_TITLE_SNIPPET,CTR_LOW_POS_4_20,Baseline heuristic; requires manual validation.,"The observed signal may be noisy, incomplete, ..."
6,REVIEW_TITLE_SNIPPET,CTR_LOW_POS_4_20,Baseline heuristic; requires manual validation.,"The observed signal may be noisy, incomplete, ..."
7,REVIEW_TITLE_SNIPPET,CTR_LOW_POS_4_20,Baseline heuristic; requires manual validation.,"The observed signal may be noisy, incomplete, ..."
8,REVIEW_TITLE_SNIPPET,CTR_LOW_POS_4_20,Baseline heuristic; requires manual validation.,"The observed signal may be noisy, incomplete, ..."
9,REVIEW_TITLE_SNIPPET,CTR_LOW_POS_4_20,Baseline heuristic; requires manual validation.,"The observed signal may be noisy, incomplete, ..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks and Leakage Check

Some high-ranked items may not represent genuine opportunities. A recommendation could be misleading if it is based on incomplete data, low impression volume, noisy click-through rates, or a search position that does not support the assumed action.

I will inspect the weakest recommendations and explain why the rule might be wrong. I will also verify that the baseline uses only information available in the observed dataset.

No product flags, future-window outcomes, or label-derived inputs are used to calculate the baseline score. The rule is a heuristic and its recommendations require further validation.


In [20]:

# SECTION 4: WEAK PICKS + LEAKAGE CHECK

# Inspect the five lowest-ranked recommendations.
weak_picks = ranked_df.tail(5).copy()

print("WEAK PICKS — BOTTOM 5")
display(weak_picks)

# Leakage check: inspect column names for possible
# future-window or label-derived inputs.
print("\nLEAKAGE CHECK")

suspicious_terms = [
    "future",
    "label",
    "target",
    "outcome",
    "product_flag"
]

suspect_columns = [
    col for col in ranked_df.columns
    if any(term in col.lower() for term in suspicious_terms)
]

if suspect_columns:
    print("Review these columns carefully:", suspect_columns)
else:
    print("No suspicious column names detected.")

print(
    "\nImportant: this is a name-based screening check. "
    "Verify the actual scoring formula and input columns "
    "manually to confirm that no leakage occurs."
)

WEAK PICKS — BOTTOM 5


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,score,reason_code,action,rank
7390,content_73eaf5c97fef,client_8527a891e2,20.0,0.00,LOW,0.00,keyword article,informational,3814.0,24499.0,...,0.0,0.0,low,striking,down,-100.0,10.004,CTR_LOW_POS_4_20,REVIEW_TITLE_SNIPPET,7391
7391,content_d58d9b1185d1,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,797.0,5741.0,...,100.0,0.0,low,striking,down,-50.0,10.003,CTR_LOW_POS_4_20,REVIEW_TITLE_SNIPPET,7392
7392,content_47d62e68cd7a,client_8527a891e2,20.0,1.00,HIGH,0.16,keyword article,transactional,3802.0,23286.0,...,0.0,0.0,low,striking,down,-100.0,10.002,CTR_LOW_POS_4_20,REVIEW_TITLE_SNIPPET,7393
7393,content_214fd3d0206d,client_8722616204,30.0,0.97,HIGH,0.14,keyword article,commercial,5543.0,39414.0,...,0.0,0.0,low,striking,flat,NaN,10.002,CTR_LOW_POS_4_20,REVIEW_TITLE_SNIPPET,7394
7394,content_ebb1a98c0f28,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,1044.0,7556.0,...,100.0,0.0,low,striking,stable,0.0,10.002,CTR_LOW_POS_4_20,REVIEW_TITLE_SNIPPET,7395



LEAKAGE CHECK
No suspicious column names detected.

Important: this is a name-based screening check. Verify the actual scoring formula and input columns manually to confirm that no leakage occurs.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.